**TO DO:**
- [x] Modificar la NN para que acepte varios canales de steps sucesivos, para añadir la dimensión temporal
- [ ] Ajustar returns function para que sea _n-steps_ reward.
- [ ] Paralelizar environments

# Actor Critic

In [ ]:
from collections import deque, namedtuple

import ale_py
import gymnasium as gym

import numpy as np

import plotly.express as px

import torch
from torch import nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils
from torch.nn.init import xavier_uniform_



GAMMA = 0.99
LEARNING_RATE = 0.002
BATCH_SIZE = 128
ENTROPY_BETA = 0.01
NUM_ENVS = 40
REWARD_STEPS = 8 # How many steps ahead we take to approximate the total discounted reward for every action.
CLIP_GRAD = 0.1 # Prevents gradients from becoming too large. If L2 norm of gradients is bigger than this number, grad vector is clipped to this value.
ADAM_EPS = 1e-3
START_TRAINING_AT = 10000
DIMS = {dim: idx for idx, dim in enumerate("env step frame height width".split())}

DEVICE = torch.device('mps')

def preprocess(obs):
    obs = obs[35:195] # crop
    obs = obs[::2,::2,0] # downsample by factor of 2
    obs[obs == 144] = 0. # erase background (background type 1)
    obs[obs == 109] = 0. # erase background (background type 2)
    obs[obs != 0] = 1. # everything else (paddles, ball) just set to 1
    obs = obs.astype(np.float32)
    obs = torch.Tensor(obs).to(DEVICE)
    return obs

def preprocess_batch(obs):
    obs = obs[:,35:195] # crop
    obs = obs[:,::2,::2,0] # downsample by factor of 2
    obs[obs == 144] = 0. # erase background (background type 1)
    obs[obs == 109] = 0. # erase background (background type 2)
    obs[obs != 0] = 1. # everything else (paddles, ball) just set to 1
    obs = obs.astype(np.float32)
    obs = torch.Tensor(obs).to(DEVICE)
    return obs

class XavierLinear(nn.Linear):
    def __init__(self, in_features: int, out_features: int, bias: bool = True, device=None, dtype=None) -> None:
        super().__init__(in_features, out_features, bias, device, dtype)
        xavier_uniform_(self.weight)

class XavierConv2d(nn.Conv2d):
    def __init__(self, in_channels: int, out_channels: int, kernel_size, stride) -> None:
        super().__init__(in_channels, out_channels, kernel_size, stride)
        xavier_uniform_(self.weight)

class A2C(nn.Module):
    def __init__(self, n_actions):
        super(A2C, self).__init__()
        self.conv = nn.Sequential(
            XavierConv2d(3, 32, kernel_size=8, stride=4), # in 80x80, out 39x39
            nn.ReLU(),
            XavierConv2d(32, 64, kernel_size=5, stride=2), # in 39x39, out 18x18
            nn.ReLU(),
            XavierConv2d(64, 64, kernel_size=2, stride=1), # in 8x8, out 17x17
            nn.ReLU()
        )
        self.flatter = nn.Flatten()
        self.policy = nn.Sequential(
            XavierLinear(3136,512), 
            nn.ReLU(),
            XavierLinear(512, n_actions)
        )
        self.value = nn.Sequential(
            XavierLinear(3136, 512), 
            nn.ReLU(),
            XavierLinear(512, 1)
        )

    def forward(self, x):
        conv_out = self.conv(x)
        flat_conv_out = conv_out.view(x.size()[0], -1)
        probs = F.softmax(self.policy(flat_conv_out), dim=1)
        value_estimate = self.value(flat_conv_out)
        return probs, value_estimate
    
def gen_run_episodes(model, envs):
    raw_state = preprocess_batch(envs.reset()[0])
    acc_state =  torch.zeros_like(raw_state[0]).tile(raw_state.shape[0], 3, 1, 1)
    acc_state[:,-1,:,:] = raw_state
    while True:
        probs, _ = model(acc_state)
        actions = probs.multinomial(1).cpu().numpy().flatten()
        obs, rewards, terminations, truncations, _ = envs.step(actions)
        yield acc_state, rewards, actions, terminations, truncations
        raw_state = preprocess_batch(obs)
        acc_state =  torch.zeros_like(raw_state[0]).tile(raw_state.shape[0], 3, 1, 1)
        acc_state[:,-1,:,:] = raw_state


def to_tensor(sratt):
    for obj in sratt:
        if not isinstance(obj, torch.Tensor):
            yield torch.Tensor(obj).to(DEVICE)
        else:
            yield obj

def discount_rewards(batch_rewards):
    # Compute discounted reward
    batch_discount_gammas  = GAMMA ** torch.arange(REWARD_STEPS).to(DEVICE)
    batch_disc_steps = batch_rewards * batch_discount_gammas
    batch_disc_rewards = batch_disc_steps.sum(dim=1)
    # Normalise
    batch_norm_disc_rewards  = (batch_disc_rewards - batch_disc_rewards.mean()) / (batch_disc_rewards.std() + 1e-10)
    return batch_norm_disc_rewards

class ReplayBuffer(deque):
    def __init__(self, maxlen=10000):
        super().__init__(maxlen=maxlen//NUM_ENVS)
        self.states = torch.Tensor().to(DEVICE)
        self.rewards = torch.Tensor().to(DEVICE)
        self.actions = torch.IntTensor().to(DEVICE)
        self.values = torch.Tensor().to(DEVICE)

    def __len__(self):
        return(NUM_ENVS*self.states.shape[DIMS["step"]])

    def append(self, sratt):
        states, rewards, actions, _, _ = to_tensor(sratt)
        states = states[:, None, :,:,:]
        rewards = rewards[:,None]
        actions = actions[:,None]
        self.states = torch.cat([self.states, states], dim=1)[:,-self.maxlen:]
        self.rewards = torch.cat([self.rewards, rewards],dim=1)[:,-self.maxlen:]
        self.actions = torch.cat([self.actions, actions],dim=1)[:,-self.maxlen:]

    
    def sample_batch(self, batch_size):
        future_rewards = self.accumulate_future_rewards()
        total_frames =  NUM_ENVS *  (self.states.shape[DIMS["step"]] - REWARD_STEPS)
        rewards_pool = future_rewards[:,:-REWARD_STEPS].reshape(total_frames, REWARD_STEPS)
        states_pool = self.states[:,:-REWARD_STEPS].reshape(
            total_frames, 
            self.states.shape[DIMS["frame"]], 
            self.states.shape[DIMS["width"]], 
            self.states.shape[DIMS["height"]], 
            )
        actions_pool = self.actions[:,:-REWARD_STEPS].reshape(-1)
        raw_rewards_pool = self.rewards[:,:-REWARD_STEPS].reshape(-1)
        idxs = np.random.choice(total_frames, size=batch_size, replace=False).tolist()
        discounted_rewards = discount_rewards(rewards_pool[idxs])
        actions = actions_pool[idxs].cpu().numpy().astype(np.int8)
        return states_pool[idxs], actions, discounted_rewards[idxs], raw_rewards_pool[idxs]
        
    
    def accumulate_future_rewards(self):
        future_rewards = self.rewards[:,:,None]
        for step in range(1, REWARD_STEPS):
            rolled = self.rewards[:,:,None].roll(-step, dims=1)
            future_rewards = torch.cat([future_rewards, rolled], dim=-1)
        return future_rewards

In [ ]:
envs = gym.make_vec("PongNoFrameskip-v4", num_envs=NUM_ENVS, vectorization_mode="async")
model = A2C(n_actions=envs.action_space[0].n).to(DEVICE)
optim = torch.optim.Adam(model.parameters(),
                         lr=LEARNING_RATE,
                         eps=ADAM_EPS)

def policy_loss_function(discounted_r, probs, actions, state_values):
    logprobs = torch.log(probs + 1e-10)
    selected = logprobs[torch.arange(probs.shape[0]), actions]    
    advantage = discounted_r - state_values.detach() # We don't want the value loss to propagate into the policy head.
    weighted = selected * advantage
    return - weighted.mean() 

def value_loss_function(discounted_r, state_values):
    return F.mse_loss(discounted_r.detach(), state_values).mean()

def entropy_loss_function(probs):
    return - (probs * (probs + 1e-10).log()).sum(dim=1).mean()

def train_model(batch):
    optim.zero_grad()
    states, actions, rewards, raw_rewards = batch
    probs, values = model(states)
    values = values.reshape(-1)
    
    policy_loss = policy_loss_function(rewards, probs, actions, values)
    value_loss = value_loss_function(rewards, values)
    entropy = entropy_loss_function(probs)
    total_loss = policy_loss + value_loss - entropy * ENTROPY_BETA
    total_loss.backward()

    nn_utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
    optim.step()
    return (loss.item() for loss in [policy_loss, value_loss, entropy, total_loss, raw_rewards.sum()])

In [ ]:
from collections import namedtuple

exp_generator = gen_run_episodes(model, envs)
rb = ReplayBuffer(maxlen=12000)
episode_rewards = np.zeros((NUM_ENVS))
episode_total_rewards = [-21]
losses = namedtuple("Losses", "policy value entropy total rewards")
losses_memory = []

def compute_episode_reward(episode_rewards, terminated):
    terminated_episodes = terminated.nonzero()[0]
    for terminated_episode in terminated_episodes:
        episode_total_rewards.append(episode_rewards[terminated_episode])
        episode_rewards[terminated_episode] = 0
    return episode_rewards

def log_progress(episode_total_rewards):
    print(f"Completed episodes: {len(episode_total_rewards)-1}.\
        Last reward: {episode_total_rewards[-1]}.\
        Mean batch reward: {results.rewards: .3f}. \
        Policy loss: {results.policy: .3f}. \
        Value loss: {results.value: .3f}. \
        Entropy loss: {results.entropy: .3f}. \
        Total loss: {results.total: .3f}. \
        ")

while np.array(episode_total_rewards)[-100:].mean() < 10:
    sratt = next(exp_generator)
    
    rewards, terminated = sratt[1], sratt[-2]
    episode_rewards += rewards
    if terminated.any():
        episode_rewards = compute_episode_reward(episode_rewards, terminated)
    
    rb.append(sratt)
    
    if len(rb) < START_TRAINING_AT:
        continue
    
    batch = rb.sample_batch(BATCH_SIZE)
    results = train_model(batch)
    results = losses(**{
            field: value 
            for field, value 
            in zip(losses._fields, results)
            })
    if results.entropy < 0.6:
        break
    losses_memory.append(results)
    log_progress(episode_total_rewards)

        




In [ ]:
import pandas as pd

lossssss = pd.DataFrame(losses_memory)
px.line(lossssss.rolling(4000).mean())

In [ ]:
px.line(episode_total_rewards
)

## Losses
### Discounted reward
For every step $t$, calculate discounted reward:

$ R = \sum_{i=0}{\gamma^{i-t} r_i} $

In [ ]:
def discount_rewards(batch_rewards):
    # Compute discounted reward
    batch_discount_gammas  = GAMMA ** torch.arange(REWARD_STEPS).to(DEVICE)
    batch_disc_steps = batch_rewards * batch_discount_gammas
    batch_disc_rewards = batch_disc_steps.sum(dim=1)
    # Normalise
    batch_norm_disc_rewards  = (batch_disc_rewards - batch_disc_rewards.mean()) / (batch_disc_rewards.std() + 1e-10)
    return batch_disc_rewards

### Policy loss
Here we discount the estimated state value from the return, as a baseline. We change the sign of the loss because we want to move towards this gradient.

$ L_\pi = -\sum_{t}log\pi_{\theta}(a_t | s_t)(R-V_{\theta}(s_t))$

### Value loss
Is a  common mean squared error, as far as I understand. Don't change sign, move opposite the direction of this gradient.

$L_v = \frac{1}{n}\sum(r-V_\theta(s_t))^2$

### Entropy loss
Helps reinforce exploration.

$L_H = - \beta\sum\pi(a|s)log\pi(a|s)$

### Total loss
We combine the three values to compute the total loss of the model.

$L = L_\pi - L_v + L_H$

## Training loop



In [ ]:
import os
def save_progress(episodes, model, rewards, probs, value_estimates, start_idxs, policy_loss, value_loss, entropy_loss, grads):
    dir = f"debug/Episode_{episodes}_data"
    os.makedirs(dir, exist_ok=True)
    torch.save(model.state_dict(), f"{dir}/model_state_dict.pt", )
    torch.save(probs, f"{dir}/probs.pt")
    torch.save(value_estimates, f"{dir}/value_estimates.pt")
    torch.save(rewards, f"{dir}/rewards.pt")
    torch.save(start_idxs, f"{dir}/start_idxs.pt")
    torch.save(torch.asarray([policy_loss, value_loss, entropy_loss]),
               f"{dir}/losses.pt")
    torch.save(torch.asarray([grads.mean().item(), grads.max().item()]),
               f"{dir}/grads.pt")